# Regioni

Analisi confrontabili tra regioni, con normalizzazioni di popolazione, superficie
e PIL regionale disponibili nei dataset.


## Scaricamento dati

Esegui questa cella prima degli import di `utils_bilancio`. La cella usa solo Python standard per trovare il repository e, se serve, lancia `scripts/run_sections.py` per creare o aggiornare il JSON della sezione.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

SECTION_ID = "regioni"
REFRESH = False
FORCE_DOWNLOAD = False
SIOPE_YEARS = ""

repo_root = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "scripts" / "utils_bilancio").is_dir():
        repo_root = candidate
        break

if repo_root is None:
    raise ModuleNotFoundError("Impossibile trovare il repository Bilancio_pubblico.")

section_file = repo_root / "data" / "export" / "bilancio-pubblico" / "sections" / f"{SECTION_ID}.json"
command = [sys.executable, str(repo_root / "scripts" / "run_sections.py"), "--sections", SECTION_ID]
if REFRESH or FORCE_DOWNLOAD:
    command.append("--refresh")

env = os.environ.copy()
if SECTION_ID == "regioni" and SIOPE_YEARS:
    env["BILANCIO_PUBBLICO_SIOPE_YEARS"] = SIOPE_YEARS

if FORCE_DOWNLOAD or REFRESH or not section_file.exists():
    print("Eseguo:", " ".join(command))
    subprocess.run(command, cwd=repo_root, check=True, env=env)
else:
    print(f"Uso cache: {section_file}")


## Import e caricamento

Dopo che i file sono presenti, questa cella importa gli helper del repo e carica `section`, `status`, `frame` e l'eventuale `source_payload`.


In [ ]:
from pathlib import Path
import sys

SECTION_ID = "regioni"

if "repo_root" not in globals() or repo_root is None:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "scripts" / "utils_bilancio").is_dir():
            repo_root = candidate
            break
    else:
        raise ModuleNotFoundError("Impossibile trovare il repository Bilancio_pubblico.")

scripts_dir = repo_root / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from utils_bilancio.notebook.utils import setup_notebook_section

notebook_state = setup_notebook_section(
    section_id=SECTION_ID,
    refresh=False,
    force_download=False,
    include_source_payload=SECTION_ID == "italia",
)

section = notebook_state["section"]
status = notebook_state["status"]
frame = notebook_state["frame"]
source_payload = notebook_state.get("source_payload")


## Parametri principali

Nella cella **Scaricamento dati** puoi modificare:
- `REFRESH = False/True`: aggiorna OpenBDAP, Eurostat regionali e SIOPE quando la sezione viene rigenerata.
- `FORCE_DOWNLOAD = False/True`: forza una nuova materializzazione anche se il JSON esiste gia'.
- `SIOPE_YEARS`: solo per SIOPE. Usa `""` per tutti gli anni previsti dal codice, `"2024"` per un test veloce, `"2019-2024"` per un intervallo, oppure `"2019,2021,2024"` per anni separati. Va impostato prima di eseguire la cella di scaricamento.

Per OpenBDAP puoi regolare:
- `MEASURE`: `"spesa"`, `"entrate"`, `"saldo"`, `"final"`.
- `METRIC`: colonna fra quelle stampate in `ITALY_REGION_METRICS`, per esempio `"mld"`, `"mld_reale"`, `"euro_per_capita"`, `"euro_reale_per_capita"`, `"euro_per_km2"`, `"pil"`.
- `YEAR`: anno fra quelli stampati in `ITALY_REGION_YEARS`; `None` usa l'ultimo disponibile.
- `REGION`: nome regione esattamente come stampato in `ITALY_REGION_OPTIONS`, ad esempio `"Lombardia"`, `"Sardegna"`, `"Toscana"`.

Per SIOPE puoi regolare:
- `SIOPE_PERIMETER`: valore fra `SIOPE_PERIMETER_OPTIONS`, ad esempio `"regioni"`, `"regioni_sanita"`, `"pa_localizzate"`.
- `SIOPE_FLOW`: `"entrate"`, `"uscite"` o `"saldo_cassa"` se presente.
- `SIOPE_METRIC`: colonna fra `SIOPE_METRICS`.
- `SIOPE_CODE`: codice gestionale fra `SIOPE_CODE_OPTIONS`, oppure `None` per scegliere automaticamente una voce rilevante.

I nomi regione, i perimetri e i codici sono case-sensitive: copia i valori dalla cella opzioni.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 170)
plt.style.use("ggplot")



In [ ]:
def rows_to_df(rows):
    return frame(rows or [])


def plot_top(df, metric, title, year=None, top=15, region_col="regione"):
    if df.empty or metric not in df.columns:
        print("Nessun dato")
        return
    if year is None and "anno" in df.columns:
        year = int(df["anno"].max())
    selected = df
    if "anno" in selected.columns and year is not None:
        selected = selected[selected["anno"] == year]
    selected = selected.dropna(subset=[metric]).sort_values(metric, ascending=False).head(top)
    if selected.empty:
        print("Nessun dato")
        return
    selected.plot(x=region_col, y=metric, kind="bar", figsize=(12, 5), legend=False)
    plt.title(f"{title} - {year if year else ''}")
    plt.tight_layout()
    plt.show()


def plot_evolution(df, region, metric, measure_label=""):
    if df.empty or metric not in df.columns:
        print("Nessun dato")
        return
    filtered = df[df["regione"] == region]
    if filtered.empty:
        print(f"Nessun dato per {region}")
        return
    filtered = filtered.sort_values("anno")
    filtered.plot(x="anno", y=metric, marker="o", figsize=(10, 4), title=f"{region} - {measure_label}")
    plt.tight_layout()
    plt.show()


def summarize_region_section(payload):
    print("copertura keys:", list(payload.keys()))
    for name, value in payload.items():
        if isinstance(value, list):
            print(f"{name}: {len(value)} righe")
        elif isinstance(value, dict):
            print(f"{name}: dict")
            for sub_name, sub_value in value.items():
                if isinstance(sub_value, list):
                    print(f"  - {name}.{sub_name}: {len(sub_value)} righe")


## Elenco opzioni disponibili

Esegui questa cella prima delle celle dove imposti parametri:

- controlla i nomi ammessi
- usa esattamente i valori indicati
- verifica i codici/regioni/anni disponibili nella tua versione dati


In [ ]:
# Opzioni confronto regionale e SIOPE - parametri ammessi

def print_values(header, values):
    print(header)
    if not values:
        print("  (nessun valore)")
        return
    for item in values:
        print(f" - {item}")


def sorted_lines(items, per_line=10):
    values = [str(item) for item in items]
    values = sorted(set(values))
    if not values:
        return ["(vuoto)"]
    return [", ".join(values[i : i + per_line]) for i in range(0, len(values), per_line)]

revenue_by_region = rows_to_df(section.get("revenue", {}).get("by_region", []))
spending_by_region = rows_to_df(section.get("spending", {}).get("by_region", []))
balances_by_region = rows_to_df(section.get("balances", {}).get("by_region", []))
final_balance = rows_to_df(section.get("balances", {}).get("final_by_region", []))
normalization_options = rows_to_df(section.get("normalization_options", []))

siope = section.get("siope", {}) or {}
siope_by_region = rows_to_df(siope.get("by_region_year", []))
siope_by_month = rows_to_df(siope.get("by_region_month", []))
siope_by_code = rows_to_df(siope.get("by_region_code_year", []))
siope_balances = rows_to_df(siope.get("balances_by_region_year", []))
siope_compartments = rows_to_df(siope.get("by_region_compartment_year", []))

openbdap_frames = [revenue_by_region, spending_by_region, balances_by_region, final_balance]
regions = set()
years = set()
for df in openbdap_frames:
    if not df.empty and "regione" in df.columns:
        regions.update(df["regione"].dropna().astype(str).tolist())
    if not df.empty and "anno" in df.columns:
        years.update(pd.to_numeric(df["anno"], errors="coerce").dropna().astype(int).tolist())

metric_candidates = ["mld", "mld_reale", "value_mld", "euro_per_capita", "euro_reale_per_capita", "euro_per_km2", "pil", "value"]
available_columns = set()
for df in openbdap_frames:
    available_columns.update(df.columns)

MEASURE_OPTIONS = ["spesa", "saldo", "final", "entrate", "altro"]
metric_options = [metric for metric in metric_candidates if metric in available_columns]

NORMALIZATION_OPTIONS = []
if not normalization_options.empty and "id" in normalization_options.columns and "label" in normalization_options.columns:
    NORMALIZATION_OPTIONS = normalization_options[["id", "label"]].to_dict("records")

by_title = rows_to_df(section.get("revenue", {}).get("by_title", []))
region_groups = []
if not by_title.empty and "regione" in by_title.columns:
    region_groups = sorted(set(by_title["regione"].dropna().astype(str).tolist()))

siope_frames = [siope_by_region, siope_by_month, siope_by_code, siope_balances, siope_compartments]
siope_regions = set()
siope_years = set()
siope_flows = set()
siope_metric_columns = set()
for df in siope_frames:
    if not df.empty and "regione" in df.columns:
        siope_regions.update(df["regione"].dropna().astype(str).tolist())
    if not df.empty and "anno" in df.columns:
        siope_years.update(pd.to_numeric(df["anno"], errors="coerce").dropna().astype(int).tolist())
    if not df.empty and "flusso" in df.columns:
        siope_flows.update(df["flusso"].dropna().astype(str).tolist())
    siope_metric_columns.update(df.columns)

perimeter_rows = siope.get("perimeters", []) or []
SIOPE_PERIMETER_OPTIONS = [item.get("id") for item in perimeter_rows if isinstance(item, dict) and item.get("id")]
SIOPE_FLOW_OPTIONS = sorted(siope_flows)
SIOPE_YEAR_OPTIONS = sorted(siope_years)
SIOPE_METRICS = [metric for metric in metric_candidates if metric in siope_metric_columns]
SIOPE_CODE_OPTIONS = []
SIOPE_CODE_LABELS = []
if not siope_by_code.empty and "codice_gestionale" in siope_by_code.columns:
    SIOPE_CODE_OPTIONS = sorted(set(siope_by_code["codice_gestionale"].dropna().astype(str).tolist()))
    if "descrizione_codice" in siope_by_code.columns:
        code_labels = siope_by_code[["codice_gestionale", "descrizione_codice"]].dropna().drop_duplicates().sort_values("codice_gestionale")
        SIOPE_CODE_LABELS = [f"{row.codice_gestionale}: {row.descrizione_codice}" for row in code_labels.itertuples(index=False)]

print("=== Parametri disponibili (OpenBDAP Regioni) ===")
print("Regioni disponibili:")
for line in sorted_lines(regions, per_line=8):
    print(f" {line}")
print(f"Totale regioni presenti: {len(regions)}")
print(f"Anni disponibili: {sorted(years)}")
print(f"Misure consigliate per MEASURE: {MEASURE_OPTIONS}")
print(f"Metriche disponibili per METRIC: {metric_options}")
print("Top consigliato: da 5 a 30")

print()
print("Normalizzazioni disponibili:")
if not NORMALIZATION_OPTIONS:
    print(" - Nessuna")
else:
    for item in NORMALIZATION_OPTIONS:
        print(f" - {item.get('id')} -> {item.get('label')}")

print()
print("REGION_GROUP possibili:")
if region_groups:
    for line in sorted_lines(region_groups, per_line=8):
        print(f" {line}")
else:
    print(" - (vuoto)")

print()
print("=== Parametri disponibili (SIOPE) ===")
print(f"Anni SIOPE caricati: {SIOPE_YEAR_OPTIONS}")
print(f"Anni completi: {siope.get('complete_years', [])}")
print(f"Anni parziali: {siope.get('partial_years', [])}")
print_values("Perimetri SIOPE:", SIOPE_PERIMETER_OPTIONS)
print_values("Flussi SIOPE:", SIOPE_FLOW_OPTIONS)
print_values("Metriche SIOPE:", SIOPE_METRICS)
print("Regioni SIOPE:")
for line in sorted_lines(siope_regions, per_line=8):
    print(f" {line}")
print(f"Codici gestionali SIOPE disponibili: {len(SIOPE_CODE_OPTIONS)}")
for line in sorted_lines(SIOPE_CODE_OPTIONS[:120], per_line=12):
    print(f" {line}")
if len(SIOPE_CODE_LABELS) > 0:
    print("Prime descrizioni codice:")
    for item in SIOPE_CODE_LABELS[:30]:
        print(f" - {item}")

print()
print("Esempi varianti:")
print(" - REGION = 'Lombardia'")
print(" - REGION = 'Sardegna'")
print(" - REGION = 'Toscana'")
print(" - NORMALIZATION = 'euro_per_capita'")
print(" - SIOPE_PERIMETER = 'regioni_sanita'")
print(" - SIOPE_FLOW = 'uscite'")
print(" - SIOPE_METRIC = 'mld_reale'")
print("Nota: nomi regione, perimetri, flussi, metriche e codici sono case-sensitive.")

ITALY_REGION_OPTIONS = sorted(regions)
ITALY_REGION_YEARS = sorted(years)
ITALY_REGION_METRICS = metric_options
ITALY_REGION_NORMALIZATIONS = [item.get("id") for item in NORMALIZATION_OPTIONS]
ITALY_REGION_GROUPS = region_groups
SIOPE_REGION_OPTIONS = sorted(siope_regions)


## Stato dati

Viene mostrato il livello di copertura e le opzioni di normalizzazione.

In [ ]:
coverage = rows_to_df(section.get("coverage", []))
normalization_options = rows_to_df(section.get("normalization_options", []))
revenue_by_region = rows_to_df(section.get("revenue", {}).get("by_region", []))
spending_by_region = rows_to_df(section.get("spending", {}).get("by_region", []))
balances_by_region = rows_to_df(section.get("balances", {}).get("by_region", []))
rev_agg = rows_to_df(section.get("revenue", {}).get("aggregates_by_region", []))
spend_agg = rows_to_df(section.get("spending", {}).get("aggregates_by_region", []))
final_balance = rows_to_df(section.get("balances", {}).get("final_by_region", []))

print("Righe copertura:", len(coverage))
print("Righe entrate:", len(revenue_by_region), "spese:", len(spending_by_region), "saldi:", len(balances_by_region))
print("Aggregati entrate:", len(rev_agg), "aggregati spese:", len(spend_agg), "saldi finali:", len(final_balance))
print("Opzioni normalizzazione:")
if not normalization_options.empty:
    display(normalization_options)
if not coverage.empty:
    display(coverage)

summarize_region_section(section)


## Classifica per anno

Scegli metrica e anno e confronta regioni.

In [ ]:
MEASURE = "spesa"
METRIC = "mld"
YEAR = None
TOP = 15

if MEASURE == "spesa":
    base = spending_by_region
elif MEASURE == "saldo":
    base = balances_by_region
else:
    base = revenue_by_region

if base.empty:
    print("Nessun dataset")
elif METRIC not in base.columns:
    print("Metrica non disponibile", METRIC)
else:
    selected_year = YEAR if YEAR is not None else int(base["anno"].max())
    selected = base[base["anno"] == selected_year]
    plot_top(selected, METRIC, f"{MEASURE} - {METRIC}", year=selected_year, top=TOP)


## Trend regionale

Analizza l'evoluzione per regione su mld, per capite o per PIL.

In [ ]:
REGION = "Lombardia"
MEASURE = "spesa"
METRIC = "mld"

if MEASURE == "spesa":
    base = spending_by_region
elif MEASURE == "saldo":
    base = balances_by_region
elif MEASURE == "final":
    base = final_balance
else:
    base = revenue_by_region

if base.empty:
    print("Nessun dato")
else:
    plot_evolution(base, REGION, METRIC, measure_label=f"{MEASURE} {METRIC}")


## Aggregati e saldo

Confronta aggregati e saldo finale con normalizzazioni multiple.

In [ ]:
for metric in ["mld", "euro_per_capita", "euro_per_km2", "pil"]:
    for title, df in [("Entrate aggregate", rev_agg), ("Spese aggregate", spend_agg)]:
        if df.empty or metric not in df.columns:
            continue
        latest_year = int(df["anno"].max())
        subset = df[df["anno"] == latest_year]
        top_rows = subset.sort_values(metric, ascending=False).head(12)
        top_rows.plot(x="regione", y=metric, kind="bar", figsize=(12, 4), legend=False)
        plt.title(f"{title} - {metric} - {latest_year}")
        plt.tight_layout()
        plt.show()

if not final_balance.empty and "mld" in final_balance.columns:
    latest_year = int(final_balance["anno"].max())
    subset = final_balance[final_balance["anno"] == latest_year].sort_values("mld", ascending=False).head(20)
    subset.plot(x="regione", y="mld", kind="bar", figsize=(12, 4), legend=False)
    plt.title(f"Saldo finale mld {latest_year}")
    plt.tight_layout()
    plt.show()

# confronto tra normalizzazioni disponibili su una misura aggregata
for metric in ["euro_per_capita", "pil", "euro_per_km2"]:
    if not rev_agg.empty and metric in rev_agg.columns:
        year = int(rev_agg["anno"].max())
        subset = rev_agg[rev_agg["anno"] == year].sort_values(metric, ascending=False).head(10)
        subset.plot(x="regione", y=metric, kind="bar", figsize=(12, 4), legend=False)
        plt.title(f"Entrate aggregate {metric} {year}")
        plt.tight_layout()
        plt.show()


## Confronti pro capite / km²

Esplora rapidamente i codici normalizzati in una singola regione e in più anni.

In [ ]:
NORMALIZATION = "euro_per_capita"
MEASURE = "spesa"
REGION_GROUP = None

if MEASURE == "spesa":
    source = spending_by_region
elif MEASURE == "saldo":
    source = balances_by_region
else:
    source = revenue_by_region

if source.empty:
    print("Nessun dato")
elif NORMALIZATION in source.columns:
    latest_year = int(source["anno"].max())
    print(f"Anno più recente: {latest_year}")
    latest = source[source["anno"] == latest_year]
    latest = latest.dropna(subset=[NORMALIZATION]).sort_values(NORMALIZATION, ascending=False).head(20)
    latest.plot(x="regione", y=NORMALIZATION, kind="bar", figsize=(12, 4), legend=False)
    plt.title(f"{MEASURE} - {NORMALIZATION} ({latest_year})")
    plt.tight_layout()
    plt.show()

    if REGION_GROUP is not None:
        region_series = source[source["regione"] == REGION_GROUP]
        if not region_series.empty:
            region_series = region_series.sort_values("anno")
            region_series.plot(x="anno", y=NORMALIZATION, marker="o", figsize=(10, 3), title=f"{REGION_GROUP} - {NORMALIZATION}")
            plt.tight_layout()
            plt.show()


## SIOPE: flussi di cassa granulari

Queste celle usano `section["siope"]`: incassi, pagamenti e saldi di cassa per regione, mese, perimetro e codice gestionale. Prima di modificarle esegui **Elenco opzioni disponibili**, poi copia da li' perimetri, flussi, metriche, anni e codici.


In [ ]:
SIOPE_PERIMETER = "regioni_sanita"
SIOPE_FLOW = "uscite"
SIOPE_METRIC = "mld"
SIOPE_YEAR = None
SIOPE_TOP = 15

siope = section.get("siope", {}) or {}
siope_by_region = rows_to_df(siope.get("by_region_year", []))
siope_balances = rows_to_df(siope.get("balances_by_region_year", []))

if SIOPE_FLOW == "saldo_cassa":
    base = siope_balances
else:
    base = siope_by_region

if base.empty:
    print("Dati SIOPE assenti: rigenera la sezione regioni con la cella di scaricamento.")
elif SIOPE_METRIC not in base.columns:
    print("Metrica SIOPE non disponibile:", SIOPE_METRIC)
else:
    selected = base.copy()
    if "perimetro" in selected.columns:
        selected = selected[selected["perimetro"] == SIOPE_PERIMETER]
    if "flusso" in selected.columns:
        selected = selected[selected["flusso"] == SIOPE_FLOW]
    if SIOPE_YEAR is None and "anno" in selected.columns and not selected.empty:
        SIOPE_YEAR = int(pd.to_numeric(selected["anno"], errors="coerce").max())
    if SIOPE_YEAR is not None and "anno" in selected.columns:
        selected = selected[selected["anno"] == SIOPE_YEAR]
    selected = selected.dropna(subset=[SIOPE_METRIC]).sort_values(SIOPE_METRIC, ascending=False).head(SIOPE_TOP)
    display(selected)
    if not selected.empty:
        selected.sort_values(SIOPE_METRIC).plot(x="regione", y=SIOPE_METRIC, kind="barh", figsize=(12, 6), legend=False)
        plt.title(f"SIOPE {SIOPE_FLOW} - {SIOPE_PERIMETER} - {SIOPE_METRIC} - {SIOPE_YEAR}")
        plt.tight_layout()
        plt.show()


In [ ]:
SIOPE_REGION = "Lombardia"
SIOPE_PERIMETER = "regioni_sanita"
SIOPE_FLOW = "uscite"
SIOPE_METRIC = "mld"
SIOPE_YEAR = None
SIOPE_CODE = None
SIOPE_TOP_CODES = 20

siope_by_code = rows_to_df((section.get("siope", {}) or {}).get("by_region_code_year", []))
if siope_by_code.empty:
    print("Dettaglio SIOPE per codice gestionale assente.")
elif SIOPE_METRIC not in siope_by_code.columns:
    print("Metrica non disponibile:", SIOPE_METRIC)
else:
    detail = siope_by_code.copy()
    detail = detail[detail["perimetro"] == SIOPE_PERIMETER]
    detail = detail[detail["flusso"] == SIOPE_FLOW]
    detail = detail[detail["regione"] == SIOPE_REGION]
    if SIOPE_YEAR is None and not detail.empty:
        SIOPE_YEAR = int(pd.to_numeric(detail["anno"], errors="coerce").max())
    if SIOPE_YEAR is not None:
        detail = detail[detail["anno"] == SIOPE_YEAR]
    if SIOPE_CODE is not None:
        detail = detail[detail["codice_gestionale"] == SIOPE_CODE]
    detail = detail.dropna(subset=[SIOPE_METRIC]).sort_values(SIOPE_METRIC, ascending=False)
    display(detail.head(SIOPE_TOP_CODES))
    if not detail.empty:
        label_col = "descrizione_codice" if "descrizione_codice" in detail.columns else "codice_gestionale"
        detail.head(SIOPE_TOP_CODES).sort_values(SIOPE_METRIC).plot(
            x=label_col, y=SIOPE_METRIC, kind="barh", figsize=(12, 8), legend=False
        )
        plt.title(f"SIOPE codici - {SIOPE_REGION} - {SIOPE_FLOW} - {SIOPE_YEAR}")
        plt.tight_layout()
        plt.show()
